# P01 (basic) — the self-model: queueing theory verified against a simulator

**Module 24 — Self-aware Computing**

A self-aware system needs a **model of itself** — something it can ask *"if the load doubles, will I
still meet my SLO?"* **before** acting. This project builds that model. You simulate a service as an
**M/M/1 queue**, measure it the way a monitoring agent would, and check the analytic predictions
against the simulation: **Little's Law**, the mean response time $R=S/(1-\rho)$, and the
**response-time quantiles** that SLOs are actually written on.

The punchline is the $1/(1-\rho)$ **explosion**: going from 50 % to 90 % utilisation costs 5x the
latency. That single non-linearity is why an auto-scaler (P02) must act on *predicted latency*, not
on a utilisation number — and why "the CPU is only at 85 %" is a dangerous sentence.

### Goal
- run a **discrete-event simulation** of a single-server queue,
- **measure** it as a monitor would ($X$, $U$, $R$) and verify **Little's Law** $N = X\cdot R$ (script ch. 4),
- implement the analytic **M/M/1** predictions and check them against the simulation (script ch. 5),
- compute **response-time quantiles** ($R_{95}$, $R_{99}$) and see why SLOs use percentiles,
- exhibit the **$1/(1-\rho)$ explosion** and answer a capacity question with the model.

### Format
Jupyter notebook — the whole point is comparing a *predicted* number to a *measured* one, side by
side, with the plot of the explosion next to it.

### Prior knowledge
The module 24 script ch. 4-5, exponential distributions, means and quantiles.

### Tasks
The simulator is given; at the `# TODO` spots you implement the measurement, the analytic predictions and the quantiles. The solution is in `solution/`.

## Setup
Only `numpy` and `matplotlib`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=4, suppress=True)

## Part A — a discrete-event simulator of the service (given)

We model one service replica as a **single server with a FIFO queue**: requests arrive with
exponential inter-arrival times (rate $\lambda$, i.e. Poisson arrivals) and each takes an
exponential service time (rate $\mu$, mean $S=1/\mu$). The core recursion is one line: a request
starts when *both* it has arrived **and** the server is free (the previous request departed).

This is our "system under observation" — the thing the self-aware system will have to model.

In [ ]:
def simulate_queue(lam, mu, n_requests=150_000, seed=0):
    """Discrete-event simulation of an M/M/1 queue. Returns per-request timing arrays."""
    rng = np.random.default_rng(seed)
    interarrival = rng.exponential(1.0 / lam, n_requests)
    service = rng.exponential(1.0 / mu, n_requests)
    arrival = np.cumsum(interarrival)

    start = np.empty(n_requests)
    depart = np.empty(n_requests)
    for i in range(n_requests):
        # a request starts when it has arrived AND the server is free
        start[i] = arrival[i] if i == 0 else max(arrival[i], depart[i - 1])
        depart[i] = start[i] + service[i]

    return dict(arrival=arrival, start=start, depart=depart, service=service)

sim = simulate_queue(lam=0.8, mu=1.0, n_requests=150_000, seed=1)
print("simulated requests:", len(sim["arrival"]))
print("first 3 response times:", (sim["depart"] - sim["arrival"])[:3].round(3))

## Part B — measure the system the way a monitor would

A monitoring agent does not know $\lambda$ or $\mu$ — it sees **completions, busy time and response
times**. From those it computes the three numbers every performance dashboard shows:

- **throughput** $X = C/T$ (completions per unit time),
- **utilisation** $U = B/T$ (busy fraction),
- **mean response time** $R$ (time from arrival to departure).

**Your task:** implement `measure(sim)`. Then we verify **Little's Law** $N = X\cdot R$ — which must
hold for *any* stable system, with no distributional assumption at all.

In [ ]:
def measure(sim):
    """What a monitoring agent observes: throughput X, utilisation U, mean response time R."""
    T = sim["depart"][-1]                 # length of the observation window
    C = len(sim["arrival"])               # completed requests
    B = sim["service"].sum()              # total busy time of the server
    # TODO: return dict(X=..., U=..., R=...)
    #   X = C / T ;  U = B / T ;  R = mean of (depart - arrival)
    raise NotImplementedError

m = measure(sim)
print(f"measured:  X = {m['X']:.4f} req/s,  U = {m['U']:.4f},  R = {m['R']:.4f} s")

# Little's Law cross-check: N = X * R must equal the true time-average number in the system
N_little = m["X"] * m["R"]
# ground truth: integrate the number in system over time (each request contributes its own R)
N_true = (sim["depart"] - sim["arrival"]).sum() / sim["depart"][-1]
print(f"Little's Law N = X*R = {N_little:.4f}   |   time-average N measured = {N_true:.4f}")
print(f"relative difference: {abs(N_little - N_true) / N_true:.2e}")

**Expectation.** Little's Law holds essentially exactly (relative difference ~$10^{-16}$) — it is
an identity, not an approximation. That is what makes it so useful: a system can measure any two of
$N, X, R$ and *know* the third, without assuming anything about how it works internally.

## Part C — the analytic self-model (M/M/1)

Now the **prediction**. Under the M/M/1 assumptions, everything follows from the utilisation
$\rho=\lambda/\mu$ (script ch. 5):

$$\rho = \frac{\lambda}{\mu},\qquad N = \frac{\rho}{1-\rho},\qquad R = \frac{1}{\mu-\lambda} = \frac{S}{1-\rho}$$

**Your task:** implement `mm1_predict(lam, mu)` returning $\rho$, $N$ and $R$ (return `None` values
or `inf` if the queue is unstable, $\rho\ge1$). Then we hold the prediction against the measurement.

In [ ]:
def mm1_predict(lam, mu):
    """Analytic M/M/1 prediction: utilisation rho, mean number N, mean response time R."""
    rho = lam / mu
    if rho >= 1.0:
        return dict(rho=rho, N=np.inf, R=np.inf)     # unstable: the queue grows without bound
    # TODO: return dict(rho=rho, N=..., R=...) using the formulas above
    raise NotImplementedError

print(f"  {'rho':>5} | {'R measured':>11} | {'R predicted':>12} | {'rel. error':>10}")
print("  " + "-" * 46)
for rho_target in [0.5, 0.7, 0.8, 0.9]:
    mu = 1.0
    lam = rho_target * mu
    s = simulate_queue(lam, mu, n_requests=150_000, seed=1)
    meas = measure(s)
    pred = mm1_predict(lam, mu)
    err = abs(meas["R"] - pred["R"]) / pred["R"]
    print(f"  {rho_target:5.2f} | {meas['R']:11.4f} | {pred['R']:12.4f} | {err:9.1%}")

**Expectation / self-check.** The prediction matches the simulation closely — well under
1 % at moderate load, growing to a few per cent at $\rho=0.9$. The model is *true*, which is what
licenses using it to make decisions.

That growth is **not** a modelling error: near saturation the queue's autocorrelation time explodes,
so a fixed-length simulation is still converging (run it with more requests and the gap shrinks).
Worth internalising — **the closer to saturation, the longer you must observe to know your own
state**, which is itself a limit on self-awareness.

## Part D — quantiles: why SLOs are written on percentiles

An SLO is almost never "mean latency < 200 ms"; it is "**95th-percentile** latency < 200 ms". The
reason is that the response-time **distribution** has a long tail. For M/M/1 the response time is
**exponentially distributed** with rate $\mu-\lambda$, so the quantiles are closed-form:

$$R_p = \frac{\ln\!\big(1/(1-p)\big)}{\mu-\lambda}$$

**Your task:** implement `response_quantile(lam, mu, p)`.

In [ ]:
def response_quantile(lam, mu, p):
    """The p-quantile of the M/M/1 response-time distribution (Exp with rate mu - lam)."""
    # TODO: return ln(1/(1-p)) / (mu - lam)
    raise NotImplementedError

lam, mu = 0.8, 1.0
s = simulate_queue(lam, mu, n_requests=150_000, seed=1)
R_samples = s["depart"] - s["arrival"]
pred = mm1_predict(lam, mu)
print(f"mean response time:      measured {R_samples.mean():7.3f}   predicted {pred['R']:7.3f}")
for p in [0.95, 0.99]:
    print(f"{int(p*100)}th percentile:        measured {np.quantile(R_samples, p):7.3f}   "
          f"predicted {response_quantile(lam, mu, p):7.3f}")
print(f"\ntail ratio R95/R : measured {np.quantile(R_samples,0.95)/R_samples.mean():.2f}  "
      f"(theory ln 20 = {np.log(20):.2f})")
print(f"tail ratio R99/R : measured {np.quantile(R_samples,0.99)/R_samples.mean():.2f}  "
      f"(theory ln 100 = {np.log(100):.2f})")

**Expectation / self-check.** $R_{95}\approx 3\times$ the mean and $R_{99}\approx 4.6\times$ the mean
— the theory says exactly $\ln 20$ and $\ln 100$. The simulated $R_{95}$ matches closely (~2.96); the
simulated $R_{99}$ comes out a little *below* theory (~4.4), because estimating a 99th percentile
needs far more samples than a mean — the deepest tail is always the least certain thing you know
about yourself.

**The consequence is operational.** A service whose *mean* latency is a comfortable 60 ms is sitting
at a 95th percentile of ~180 ms and a 99th of ~280 ms. Reporting the mean would hide the fact that
1 in 100 users waits nearly five times longer. A self-aware system must model the **tail**, because
that is what its SLO is about.

## Part E — the $1/(1-\rho)$ explosion, and a capacity question (given)

Everything above converges on one number: the factor $1/(1-\rho)$. We tabulate and plot it, then use
the model the way a self-aware system would — to answer a **capacity question before acting**.

In [ ]:
print("  the utilisation explosion (R in units of the service time S):")
print(f"  {'rho':>6} | {'R/S':>8}")
print("  " + "-" * 18)
for rho in [0.1, 0.5, 0.8, 0.9, 0.95, 0.99]:
    print(f"  {rho:6.2f} | {1/(1-rho):8.1f}")

rhos = np.linspace(0.01, 0.97, 300)
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.2))
ax1.plot(rhos, 1 / (1 - rhos), color="crimson", lw=2)
ax1.axvline(0.8, ls="--", color="gray"); ax1.axvline(0.9, ls=":", color="gray")
ax1.set_xlabel("utilisation rho"); ax1.set_ylabel("R / S")
ax1.set_title("response time explodes as rho -> 1"); ax1.grid(alpha=0.3); ax1.set_ylim(0, 35)

# the same in the units an SLO cares about: the 95th percentile
mu = 1.0
lams = rhos * mu
ax2.plot(rhos, [response_quantile(l, mu, 0.95) for l in lams], color="steelblue", lw=2, label="R95")
ax2.plot(rhos, [mm1_predict(l, mu)["R"] for l in lams], color="black", lw=1.2, ls="--", label="mean R")
ax2.set_xlabel("utilisation rho"); ax2.set_ylabel("latency (units of S)")
ax2.set_title("the tail is ~3x the mean, at every load"); ax2.legend(); ax2.grid(alpha=0.3)
ax2.set_ylim(0, 100)
plt.tight_layout(); plt.show()

In [ ]:
# A capacity question, answered by the model instead of by trial and error:
# "Requests arrive at 80/s, one replica serves 100/s. My SLO is a 95th-percentile latency
#  below 200 ms. Do I meet it? If not, what service rate would I need?"
lam = 90.0
mu = 100.0
slo = 0.200

r95 = response_quantile(lam, mu, 0.95)
print(f"at lam={lam}/s, mu={mu}/s:  rho = {lam/mu:.2f},  mean R = {mm1_predict(lam,mu)['R']*1000:.0f} ms,"
      f"  R95 = {r95*1000:.0f} ms")
print(f"SLO (R95 < {slo*1000:.0f} ms) met? {r95 < slo}")

# solve R95 = ln(20)/(mu - lam) = slo  =>  mu = lam + ln(20)/slo
mu_needed = lam + np.log(20) / slo
print(f"\nservice rate needed to meet the SLO: mu >= {mu_needed:.1f}/s "
      f"(currently {mu}/s -> need {mu_needed/mu:.2f}x the capacity)")
print(f"check: R95 at mu={mu_needed:.1f} is {response_quantile(lam, mu_needed, 0.95)*1000:.0f} ms")

**Expectation / self-check.** At $\lambda=90$, $\mu=100$ the *mean* latency is a comfortable
100 ms — but $R_{95}\approx\mathbf{300}$ ms, so the 200 ms SLO is **violated**. Judging this system by
its mean would have declared it healthy.

The inverted question is answered in closed form: to hold $R_{95}<200$ ms at $\lambda=90$/s you need
$\mu \ge \lambda + \ln(20)/\mathrm{SLO} \approx \mathbf{105}$/s — just **1.05x** the capacity to cut
the tail from 300 ms to 200 ms. The $1/(1-\rho)$ non-linearity cuts *both* ways: near saturation a
tiny capacity increase buys a disproportionate latency improvement, which is exactly why a
well-timed small scaling action is so effective.

**This is the self-predictive property in miniature.** The system did not have to *try* a
configuration and observe the damage; it **computed the consequence in advance** from a model of
itself. That is exactly what the auto-scaler in P02 will do, once per control interval.

## Conclusion

You have built the **self-model** that the rest of the module rests on:
- a **simulator** of the service, and the **measurements** a monitor would take,
- **Little's Law** $N=X\cdot R$ verified to machine precision — an identity that holds for any system,
- the analytic **M/M/1** predictions, matching the simulation to ~1-3 % (and you saw *why* the error
  grows near saturation),
- the **response-time quantiles**, with $R_{95}\approx3R$, $R_{99}\approx4.6R$ — the reason SLOs are percentiles,
- and the **$1/(1-\rho)$ explosion**, used to answer a capacity question *before* acting.

**P02 (medium)** puts this model inside a **MAPE-K loop**: three auto-scaling policies — reactive,
control-theoretic and model-based predictive — measured against each other on SLO violations, cost
and flapping. The predictive one uses exactly the formulas you just verified.